# Project1: Creating a Model
For Project Check-in #3. Use MOMP Data

### Setup imports

In [27]:
# !pip install pandas
# !pip install scikit-learn
# !pip install matplotlib

In [28]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import numpy as np

In [29]:
MOMP_df_Orig = pd.read_csv('Data/MOMP_Data_KL_2019.csv')

MOMP_df_Orig.shape

(15840, 16)

In [30]:
MOMP_df_Orig.head()

,Unnamed: 0,species,species.code,stop,date,year,occ.hist,julian.z,time.m.z,cloud.cover.z,wind.z,noise.z,snow.cover.z,precip.z,temp.z,lat.z
0,5011,Barred Owl,BARR,1,2004-03-05,2004,10110000000,-2.888471,-0.813241,0.0,0.485932,-0.59775,1.552534,-0.181054,0.40585,0.0
1,5012,Great Horned Owl,GHOW,1,2004-03-05,2004,0,-2.888471,-0.813241,0.0,0.485932,-0.59775,1.552534,-0.181054,0.40585,0.0
2,5013,Northern Saw-whet Owl,NSWO,1,2004-03-05,2004,0,-2.888471,-0.813241,0.0,0.485932,-0.59775,1.552534,-0.181054,0.40585,0.0
3,5014,Barred Owl,BARR,2,2004-03-05,2004,0,-2.888471,-0.652362,0.0,0.485932,-0.59775,0.000000,-0.181054,0.40585,0.0
4,5015,Great Horned Owl,GHOW,2,2004-03-05,2004,10000000000000,-2.888471,-0.652362,0.0,0.485932,-0.59775,0.000000,-0.181054,0.40585,0.0


### Alter so that the Dataframe only consists of the desired inputs and ouput
Inputs: Wind, Noise, Temperature, Cloud Cover, and Time since Midnight (time.m.z)
Output: Occupancy History (occ.hist)

In [31]:
df_inp_and_outp = MOMP_df_Orig.drop(MOMP_df_Orig.columns[[0, 1, 2, 3, 4, 5, 7, 12, 13, 15]], axis=1)


print(df_inp_and_outp.shape)
df_inp_and_outp.head()

(15840, 6)


,occ.hist,time.m.z,cloud.cover.z,wind.z,noise.z,temp.z
0,10110000000,-0.813241,0.0,0.485932,-0.59775,0.40585
1,0,-0.813241,0.0,0.485932,-0.59775,0.40585
2,0,-0.813241,0.0,0.485932,-0.59775,0.40585
3,0,-0.652362,0.0,0.485932,-0.59775,0.40585
4,10000000000000,-0.652362,0.0,0.485932,-0.59775,0.40585


### Only use first 5000 rows for shorter times
Note: Many zeros, tried interpolating with undesired results. Will try one hot encoding with the original dataframe later. Also worth noting, the occ.hist is recorded in binary, so we will probably have to convert the occ.hist data to the numbers they represent. I suspect we'll run into an issue on why they were recorded in binary in the first place.

In [32]:
df_updated = df_inp_and_outp.head(5000)
df_updated.head(20)

,occ.hist,time.m.z,cloud.cover.z,wind.z,noise.z,temp.z
0,10110000000,-0.813241,0.0,0.485932,-0.59775,0.40585
1,0,-0.813241,0.0,0.485932,-0.59775,0.40585
2,0,-0.813241,0.0,0.485932,-0.59775,0.40585
3,0,-0.652362,0.0,0.485932,-0.59775,0.40585
4,10000000000000,-0.652362,0.0,0.485932,-0.59775,0.40585
5,110111111001,-0.652362,0.0,0.485932,-0.59775,0.40585
6,1,-0.557727,0.0,0.485932,1.05416,0.40585
7,0,-0.557727,0.0,0.485932,1.05416,0.40585
8,0,-0.557727,0.0,0.485932,1.05416,0.40585
9,0,0.000000,0.0,0.000000,0.00000,0.00000


### Separate ML input and output

In [33]:
X = df_updated.drop('occ.hist', axis = 1)
y = df_updated['occ.hist']

display(X.head())
display(y.head())

,time.m.z,cloud.cover.z,wind.z,noise.z,temp.z
0,-0.813241,0.0,0.485932,-0.59775,0.40585
1,-0.813241,0.0,0.485932,-0.59775,0.40585
2,-0.813241,0.0,0.485932,-0.59775,0.40585
3,-0.652362,0.0,0.485932,-0.59775,0.40585
4,-0.652362,0.0,0.485932,-0.59775,0.40585


0       10110000000
1                 0
2                 0
3                 0
4    10000000000000
Name: occ.hist, dtype: int64

### Train-test split

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 20% of the data goes toward testing, other 80% goes to training

X_train.shape

(4000, 5)

### Make a Model, train it, and evaluate for accuracy.
Initial thoughts: Accuracy is probably not going to be high, as I'm going to do the grid search later for time's sake. Using all the default hyperparams for RFC. 

In [35]:
model1 = RandomForestClassifier()

model1.fit(X_train, y_train)

y_pred = model1.predict(X_test)

y_pred

array([               0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,   10110000100000,  100000000000110,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,                0,
                      0,                0,        110000000,
                      0,                0,                0,
                      0,                0,                0,
                      0,

### Computing Accuracy
Pleasantly surprised by how high the predictions managed to be, I thought it would be 40% accuracy or less, but ended up reaching 79.7% accuracy.

In [36]:
model1.score(X_test, y_test)

0.797